#### Import numpy & keras

In [1]:
import numpy as np
import keras
from datasets import load_dataset, DatasetDict, Image, Dataset
import datetime

In [2]:
import tensorflow as tf
tf.config.optimizer.set_jit(False)
print(tf.__version__)
print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))
print(tf.config.list_physical_devices('GPU'))

2.21.0
Num GPUs Available:  0
[]


### Bilder normalisieren

In [3]:
def transform(example):
    image = np.array(example["image"], dtype=np.float32) / 255.0
    return {"image": image, "label": example["label"]}


#### 1. get training data

In [4]:
import matplotlib.pyplot as plt
import PIL
print(PIL.__version__)

ds = load_dataset("jonathan-roberts1/NWPU-RESISC45")
print(ds.shape)

train_data = ds["train"]
split_1 = train_data.train_test_split(
    test_size=0.15,
    seed=42,          # sorgt dafür, dass Validation immer gleich bleibt
    shuffle=True
)

validation_dataset = split_1["test"]

12.2.0


{'train': (31500, 2)}


In [5]:
remaining_dataset = split_1["train"]
split_2 = remaining_dataset.train_test_split(
    test_size=0.25,
    seed=42,
    shuffle=True
)

train_ds = split_2["train"]

In [6]:
from datasets import concatenate_datasets
import random

# -----------------------------
# AUGMENTER (keras statt tf.keras)
# -----------------------------
augmenter = keras.Sequential([
    keras.layers.RandomFlip("horizontal_and_vertical"),
    keras.layers.RandomRotation(0.5),
    keras.layers.RandomZoom(height_factor=(-0.1, 0.1), width_factor=(-0.1, 0.1)),
    keras.layers.RandomTranslation(height_factor=0.1, width_factor=0.1),
    keras.layers.RandomContrast(0.15),
])

# -----------------------------
# AUGMENTATIONSANTEIL
# -----------------------------
augment_fraction = 0.2
num_augmented = int(len(train_ds) * augment_fraction)

indices = random.sample(range(len(train_ds)), num_augmented)
subset_to_augment = train_ds.select(indices)

# -----------------------------
# BATCH AUGMENTATION FUNKTION
# -----------------------------
def augment_batch(batch):
    images = np.array(batch["image"], dtype=np.float32)

    images = keras.ops.convert_to_tensor(images)
    images = augmenter(images, training=True)

    images = keras.ops.clip(images, 0, 255)
    images = keras.ops.convert_to_numpy(images).astype(np.uint8)

    return {
        "image": images,
        "label": np.array(batch["label"], dtype=np.int32)
    }

# -----------------------------
# MAP (batch processing)
# -----------------------------
augmented_ds = subset_to_augment.map(
    augment_batch,
    batched=True,
    batch_size=32
)

# -----------------------------
# CONCATENATE
# -----------------------------
train_ds = concatenate_datasets([train_ds, augmented_ds])

# -----------------------------
# OUTPUT
# -----------------------------
print("Originale Trainingsdaten:", len(split_2["train"]))
print("Nach Augmentierung:", len(train_ds))

Map:   0%|          | 0/4016 [00:00<?, ? examples/s]

C:\Users\marie\AppData\Local\Programs\Python\Python312\Lib\site-packages\datasets\features\image.py:397: UserWarning: Downcasting array dtype int64 to uint8 to be compatible with 'Pillow'
  warnings.warn(f"Downcasting array dtype {dtype} to {dest_dtype} to be compatible with 'Pillow'")


Originale Trainingsdaten: 20081
Nach Augmentierung: 24097


In [7]:
from datasets import DatasetDict

test_ds = split_2["test"]

# DatasetDict erzeugen
final_dataset = DatasetDict({
    "train": train_ds,
    "validation": validation_dataset,
    "test": test_ds
})

# Optional nur falls image-feature kaputt ist
# from datasets import Image
# final_dataset = final_dataset.cast_column("image", Image())

# Transformation anwenden
final_dataset = final_dataset.with_transform(transform)

print(final_dataset)

# TensorFlow Datasets
tf_train = final_dataset["train"].to_tf_dataset(
    columns="image",
    label_cols="label",
    batch_size=128,
    shuffle=True
)

tf_test = final_dataset["test"].to_tf_dataset(
    columns="image",
    label_cols="label",
    batch_size=128
)

# Klassen
class_names = final_dataset["train"].features["label"].names
print(class_names)

# Bildform prüfen
img_shape = np.array(final_dataset["train"][0]["image"]).shape
print(img_shape)

DatasetDict({
    train: Dataset({
        features: ['image', 'label'],
        num_rows: 24097
    })
    validation: Dataset({
        features: ['image', 'label'],
        num_rows: 4725
    })
    test: Dataset({
        features: ['image', 'label'],
        num_rows: 6694
    })
})
['airplane', 'airport', 'baseball diamond', 'basketball court', 'beach', 'bridge', 'chaparral', 'church', 'circular farmland', 'cloud', 'commercial area', 'dense residential', 'desert', 'forest', 'freeway', 'golf course', 'ground track field', 'harbor', 'industrial area', 'intersection', 'island', 'lake', 'meadow', 'medium residential', 'mobile home park', 'mountain', 'overpass', 'palace', 'parking lot', 'railway', 'railway station', 'rectangular farmland', 'river', 'roundabout', 'runway', 'sea ice', 'ship', 'snowberg', 'sparse residential', 'stadium', 'storage tank', 'tennis court', 'terrace', 'thermal power station', 'wetland']
(256, 256, 3)


#### 2. define architecture

In [23]:
# Import a model from /models/*
## Todo: adjust modelname
from models.leNet_5 import generateModel
model_name = "leNet_5"

load model

In [24]:
model = generateModel(img_shape)
#model = keras.models.load_model("./models/" + model_name + ".keras")
model.summary()

C:\Users\marie\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_6"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 252, 252, 6)    │           456 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ average_pooling2d               │ (None, 126, 126, 6)    │             0 │
│ (AveragePooling2D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 122, 122, 16)   │         2,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ average_pooling2d_1             │ (None, 61, 61, 16)     │             0 │
│ (AveragePooling2D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 57, 57, 32)     │        12,832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ average_pooling2d_2             │ (None, 28, 28, 32)     │             0 │
│ (AveragePooling2D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 24, 24, 32)     │        25,632 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ average_pooling2d_3             │ (None, 12, 12, 32)     │             0 │
│ (AveragePooling2D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 8, 8, 32)       │        25,632 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ average_pooling2d_4             │ (None, 4, 4, 32)       │             0 │
│ (AveragePooling2D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 512)            │       262,656 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 128)            │        65,664 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 45)             │         5,805 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 401,093 (1.53 MB)

 Trainable params: 401,093 (1.53 MB)

 Non-trainable params: 0 (0.00 B)

#### 3. set training parameter and fit model

non-specific callbacks

In [8]:
early_stopping = keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

reduce_lr = keras.callbacks.ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.3,
    patience=2,
    verbose=1,
    min_lr=1e-6
)

In [26]:
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

log_dir = "logs/" + model_name + "/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorboard_callback = tf.keras.callbacks.TensorBoard(log_dir=log_dir, histogram_freq=1)

savepath = "./models/" + model_name + ".keras"
checkpoint_callback = keras.callbacks.ModelCheckpoint(
    filepath=savepath,
    save_freq="epoch",
    save_best_only=False,
    verbose=1
)

model.fit(
    tf_train,
    validation_data=tf_test,
    epochs=40,
    callbacks=[tensorboard_callback, early_stopping, reduce_lr],
)

model.save(savepath)

Epoch 1/40
189/189 ━━━━━━━━━━━━━━━━━━━━ 57s 293ms/step - accuracy: 0.1593 - loss: 3.1287 - val_accuracy: 0.2139 - val_loss: 2.8806 - learning_rate: 0.0010
Epoch 2/40
189/189 ━━━━━━━━━━━━━━━━━━━━ 56s 294ms/step - accuracy: 0.2325 - loss: 2.8156 - val_accuracy: 0.1839 - val_loss: 3.0358 - learning_rate: 0.0010
Epoch 3/40
189/189 ━━━━━━━━━━━━━━━━━━━━ 55s 289ms/step - accuracy: 0.2600 - loss: 2.6904 - val_accuracy: 0.2599 - val_loss: 2.7507 - learning_rate: 0.0010
Epoch 4/40
189/189 ━━━━━━━━━━━━━━━━━━━━ 54s 284ms/step - accuracy: 0.2906 - loss: 2.5648 - val_accuracy: 0.2737 - val_loss: 2.6388 - learning_rate: 0.0010
Epoch 5/40
189/189 ━━━━━━━━━━━━━━━━━━━━ 53s 282ms/step - accuracy: 0.3168 - loss: 2.4632 - val_accuracy: 0.3177 - val_loss: 2.4820 - learning_rate: 0.0010
Epoch 6/40
189/189 ━━━━━━━━━━━━━━━━━━━━ 55s 291ms/step - accuracy: 0.3524 - loss: 2.3206 - val_accuracy: 0.3219 - val_loss: 2.4505 - learning_rate: 0.0010
Epoch 7/40
189/189 ━━━━━━━━━━━━━━━━━━━━ 55s 293ms/step - accuracy: 0.3

# Model 2 - four-block-cnn

In [27]:
## Todo: adjust modelname
from models.four_block_cnn import generateModel
model_name = "four_block_cnn"
model = generateModel(img_shape)
#model = keras.models.load_model("./models/" + model_name + ".keras")
model.summary()
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

log_dir = "logs/" + model_name + "/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorboard_callback = tf.keras.callbacks.TensorBoard(log_dir=log_dir, histogram_freq=1)

savepath = "./models/" + model_name + ".keras"
checkpoint_callback = keras.callbacks.ModelCheckpoint(
    filepath=savepath,
    save_freq="epoch",
    save_best_only=False,
    verbose=1
)

model.fit(
    tf_train,
    validation_data=tf_test,
    epochs=15,
    callbacks=[tensorboard_callback, early_stopping, reduce_lr],
)

model.save(savepath)

Model: "sequential_7"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_5 (Conv2D)               │ (None, 256, 256, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 256, 256, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 128, 128, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_6 (Conv2D)               │ (None, 128, 128, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 128, 128, 64)   │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 64, 64, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_7 (Conv2D)               │ (None, 64, 64, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 64, 64, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 32, 32, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_8 (Conv2D)               │ (None, 32, 32, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 32, 32, 256)    │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 16, 16, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_9 (Conv2D)               │ (None, 16, 16, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 16, 16, 256)    │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ (None, 8, 8, 256)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 256)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 256)            │        65,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 45)             │        11,565 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,058,797 (4.04 MB)

 Trainable params: 1,057,325 (4.03 MB)

 Non-trainable params: 1,472 (5.75 KB)

Epoch 1/15
189/189 ━━━━━━━━━━━━━━━━━━━━ 420s 2s/step - accuracy: 0.2916 - loss: 2.5901 - val_accuracy: 0.0367 - val_loss: 5.5198 - learning_rate: 0.0010
Epoch 2/15
189/189 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.4326 - loss: 1.9751
Epoch 2: ReduceLROnPlateau reducing learning rate to 0.0003000000142492354.
189/189 ━━━━━━━━━━━━━━━━━━━━ 419s 2s/step - accuracy: 0.4577 - loss: 1.8757 - val_accuracy: 0.0532 - val_loss: 5.5456 - learning_rate: 0.0010
Epoch 3/15
189/189 ━━━━━━━━━━━━━━━━━━━━ 417s 2s/step - accuracy: 0.5685 - loss: 1.4503 - val_accuracy: 0.3639 - val_loss: 2.4194 - learning_rate: 3.0000e-04
Epoch 4/15
189/189 ━━━━━━━━━━━━━━━━━━━━ 409s 2s/step - accuracy: 0.6136 - loss: 1.2941 - val_accuracy: 0.5995 - val_loss: 1.3572 - learning_rate: 3.0000e-04
Epoch 5/15
189/189 ━━━━━━━━━━━━━━━━━━━━ 401s 2s/step - accuracy: 0.6497 - loss: 1.1635 - val_accuracy: 0.6506 - val_loss: 1.1828 - learning_rate: 3.0000e-04
Epoch 6/15
189/189 ━━━━━━━━━━━━━━━━━━━━ 401s 2s/step - accuracy: 0.6875 -

# Model 3 - five_block_v1

In [9]:
## Todo: adjust modelname
from models.five_block_v1 import generateModel
model_name = "five_block_v1"
model = generateModel(img_shape)
#model = keras.models.load_model("./models/" + model_name + ".keras")
model.summary()
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

log_dir = "logs/" + model_name + "/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorboard_callback = tf.keras.callbacks.TensorBoard(log_dir=log_dir, histogram_freq=1)

savepath = "./models/" + model_name + ".keras"
checkpoint_callback = keras.callbacks.ModelCheckpoint(
    filepath=savepath,
    save_freq="epoch",
    save_best_only=False,
    verbose=1
)

model.fit(
    tf_train,
    validation_data=tf_test,
    epochs=20,
    callbacks=[tensorboard_callback, early_stopping, reduce_lr],
)

model.save(savepath)

C:\Users\marie\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 256, 256, 16)   │           448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 128, 128, 16)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 128, 128, 40)   │        16,040 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 128, 128, 40)   │           160 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ spatial_dropout2d               │ (None, 128, 128, 40)   │             0 │
│ (SpatialDropout2D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 64, 64, 40)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 64, 64, 80)     │        28,880 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ average_pooling2d               │ (None, 32, 32, 80)     │             0 │
│ (AveragePooling2D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 32, 32, 160)    │       115,360 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 16, 16, 160)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 16, 16, 288)    │       415,008 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 16, 16, 288)    │         1,152 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ spatial_dropout2d_1             │ (None, 16, 16, 288)    │             0 │
│ (SpatialDropout2D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 8, 8, 288)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 288)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 384)            │       110,976 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 384)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 45)             │        17,325 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 705,349 (2.69 MB)

 Trainable params: 704,693 (2.69 MB)

 Non-trainable params: 656 (2.56 KB)

Epoch 1/20
189/189 ━━━━━━━━━━━━━━━━━━━━ 160s 841ms/step - accuracy: 0.2141 - loss: 2.8907 - val_accuracy: 0.0429 - val_loss: 6.6569 - learning_rate: 0.0010
Epoch 2/20
189/189 ━━━━━━━━━━━━━━━━━━━━ 165s 875ms/step - accuracy: 0.3626 - loss: 2.2225 - val_accuracy: 0.0601 - val_loss: 5.6982 - learning_rate: 0.0010
Epoch 3/20
189/189 ━━━━━━━━━━━━━━━━━━━━ 164s 868ms/step - accuracy: 0.4483 - loss: 1.8898 - val_accuracy: 0.3446 - val_loss: 2.4285 - learning_rate: 0.0010
Epoch 4/20
189/189 ━━━━━━━━━━━━━━━━━━━━ 163s 863ms/step - accuracy: 0.5111 - loss: 1.6417 - val_accuracy: 0.4873 - val_loss: 1.8044 - learning_rate: 0.0010
Epoch 5/20
189/189 ━━━━━━━━━━━━━━━━━━━━ 165s 872ms/step - accuracy: 0.5692 - loss: 1.4419 - val_accuracy: 0.5500 - val_loss: 1.5233 - learning_rate: 0.0010
Epoch 6/20
189/189 ━━━━━━━━━━━━━━━━━━━━ 164s 869ms/step - accuracy: 0.6174 - loss: 1.2705 - val_accuracy: 0.6174 - val_loss: 1.2568 - learning_rate: 0.0010
Epoch 7/20
189/189 ━━━━━━━━━━━━━━━━━━━━ 164s 869ms/step - accura

# Model 4 - five_block_v2

In [13]:
## Todo: adjust modelname
from models.five_block_v2 import generateModel
model_name = "five_block_v2"
model = generateModel(img_shape)
#model = keras.models.load_model("./models/" + model_name + ".keras")
model.summary()
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

log_dir = "logs/" + model_name + "/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorboard_callback = tf.keras.callbacks.TensorBoard(log_dir=log_dir, histogram_freq=1)

savepath = "./models/" + model_name + ".keras"
checkpoint_callback = keras.callbacks.ModelCheckpoint(
    filepath=savepath,
    save_freq="epoch",
    save_best_only=False,
    verbose=1
)

model.fit(
    tf_train,
    validation_data=tf_test,
    epochs=20,
    callbacks=[tensorboard_callback, early_stopping, reduce_lr],
)

model.save(savepath)

Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_20 (Conv2D)              │ (None, 256, 256, 24)   │         1,800 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_17          │ (None, 256, 256, 24)   │            96 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_18 (Activation)      │ (None, 256, 256, 24)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_19 (MaxPooling2D) │ (None, 128, 128, 24)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_21 (Conv2D)              │ (None, 128, 128, 48)   │        10,368 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_18          │ (None, 128, 128, 48)   │           192 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_19 (Activation)      │ (None, 128, 128, 48)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_20 (MaxPooling2D) │ (None, 64, 64, 48)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_22 (Conv2D)              │ (None, 64, 64, 96)     │        41,472 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_19          │ (None, 64, 64, 96)     │           384 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_20 (Activation)      │ (None, 64, 64, 96)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_21 (MaxPooling2D) │ (None, 32, 32, 96)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_23 (Conv2D)              │ (None, 32, 32, 192)    │       165,888 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_20          │ (None, 32, 32, 192)    │           768 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_21 (Activation)      │ (None, 32, 32, 192)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ spatial_dropout2d_5             │ (None, 32, 32, 192)    │             0 │
│ (SpatialDropout2D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_22 (MaxPooling2D) │ (None, 16, 16, 192)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_24 (Conv2D)              │ (None, 16, 16, 320)    │       552,960 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_21          │ (None, 16, 16, 320)    │         1,280 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_22 (Activation)      │ (None, 16, 16, 320)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_23 (MaxPooling2D) │ (None, 8, 8, 320)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_max_pooling2d_3          │ (None, 320)            │             

 Total params: 868,949 (3.31 MB)

 Trainable params: 867,589 (3.31 MB)

 Non-trainable params: 1,360 (5.31 KB)

Epoch 1/20
189/189 ━━━━━━━━━━━━━━━━━━━━ 310s 2s/step - accuracy: 0.2995 - loss: 2.5814 - val_accuracy: 0.0620 - val_loss: 5.2626 - learning_rate: 0.0010
Epoch 2/20
189/189 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.4669 - loss: 1.8660
Epoch 2: ReduceLROnPlateau reducing learning rate to 0.0003000000142492354.
189/189 ━━━━━━━━━━━━━━━━━━━━ 311s 2s/step - accuracy: 0.4895 - loss: 1.7659 - val_accuracy: 0.3269 - val_loss: 2.6469 - learning_rate: 0.0010
Epoch 3/20
189/189 ━━━━━━━━━━━━━━━━━━━━ 311s 2s/step - accuracy: 0.5998 - loss: 1.3515 - val_accuracy: 0.5605 - val_loss: 1.4975 - learning_rate: 3.0000e-04
Epoch 4/20
189/189 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.6305 - loss: 1.2343
Epoch 4: ReduceLROnPlateau reducing learning rate to 9.000000427477062e-05.
189/189 ━━━━━━━━━━━━━━━━━━━━ 311s 2s/step - accuracy: 0.6372 - loss: 1.2185 - val_accuracy: 0.6150 - val_loss: 1.3396 - learning_rate: 3.0000e-04
Epoch 5/20
189/189 ━━━━━━━━━━━━━━━━━━━━ 316s 2s/step - accuracy: 0.6799 - loss: 

# Model 5 - resNet

In [30]:
## Todo: adjust modelname
from models.resNet import generateModel
model_name = "resNet"
model = generateModel(img_shape)
#model = keras.models.load_model("./models/" + model_name + ".keras")
model.summary()
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

log_dir = "logs/" + model_name + "/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorboard_callback = tf.keras.callbacks.TensorBoard(log_dir=log_dir, histogram_freq=1)

savepath = "./models/" + model_name + ".keras"
checkpoint_callback = keras.callbacks.ModelCheckpoint(
    filepath=savepath,
    save_freq="epoch",
    save_best_only=False,
    verbose=1
)

model.fit(
    tf_train,
    validation_data=tf_test,
    epochs=5,
    callbacks=[tensorboard_callback, early_stopping, reduce_lr],
)

model.save(savepath)

Model: "functional_75"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_10      │ (None, 256, 256,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_20 (Conv2D)  │ (None, 256, 256,  │        896 │ input_layer_10[0… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 256, 256,  │        128 │ conv2d_20[0][0]   │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_21 (Conv2D)  │ (None, 256, 256,  │      9,248 │ batch_normalizat… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 256, 256,  │        128 │ conv2d_21[0][0]   │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation          │ (None, 256, 256,  │          0 │ batch_normalizat… │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_22 (Conv2D)  │ (None, 256, 256,  │      9,248 │ activation[0][0]  │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 256, 256,  │        128 │ conv2d_22[0][0]   │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 256, 256,  │          0 │ batch_normalizat… │
│                     │ 32)               │            │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_1        │ (None, 256, 256,  │          0 │ add[0][0]         │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_23 (Conv2D)  │ (None, 256, 256,  │      9,248 │ activation_1[0][… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 256, 256,  │        128 │ conv2d_23[0][0]   │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_2        │ (None, 256, 256,  │          0 │ batch_normalizat… │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_24 (Conv2D)  │ (None, 256, 256,  │      9,248 │ activation_2[0][… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 256, 256,  │        128 │ conv2d_24[0][0]   │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_1 (Add)         │ (None, 256, 256,  │          0 │ batch_normalizat… │
│                     │ 32)               │            │ activation_1[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_3        │ (None, 256, 256,  │          0 │ add_1[0][0]     

 Total params: 2,879,597 (10.98 MB)

 Trainable params: 2,874,797 (10.97 MB)

 Non-trainable params: 4,800 (18.75 KB)

Epoch 1/5
109/189 ━━━━━━━━━━━━━━━━━━━━ 19:35 15s/step - accuracy: 0.1335 - loss: 3.4404

KeyboardInterrupt: 

In [19]:
#%load_ext tensorboard
%reload_ext tensorboard
%tensorboard --logdir logs

Reusing TensorBoard on port 6006 (pid 33128), started 2 days, 8:23:13 ago. (Use '!kill 33128' to kill it.)

#### 4. predict output 

In [ ]:
import random

# Zufälligen Index wählen
idx = random.randint(0, len(final_dataset["test"]) - 1)

# Sample holen
sample = final_dataset["test"][idx]

# Bild und echtes Label
img = sample["image"]
true_idx = sample["label"]

# Batch-Dimension hinzufügen
input_img = np.expand_dims(img, axis=0)

# Prediction
prediction = model.predict(input_img, verbose=0)

# Vorhersageklasse
predicted_idx = np.argmax(prediction)

# Ausgabe
print("Index:", idx)
print("Predicted:", class_names[predicted_idx])
print("True:", class_names[true_idx])

# Bild anzeigen
plt.imshow(img)
plt.title(f"Pred: {class_names[predicted_idx]} | True: {class_names[true_idx]}")
plt.axis("off")
plt.show()